In [ ]:
# ── Colab Setup ───────────────────────────────────────────────────────────────
# Run this cell first every time you open a new Colab session.
# Clones the repo (data and artifacts included) and configures the environment.
import os, sys

REPO_URL  = "https://github.com/tackes/Modern-Time-Series-Forecasting-Cohort.git"
REPO_PATH = "/content/packt-modern-time-series"

if not os.path.exists(REPO_PATH):
    os.system(f"git clone -q {REPO_URL} {REPO_PATH}")

# Stay in instructor_notebooks so Path().resolve().parent resolves to repo root
os.chdir(f"{REPO_PATH}/instructor_notebooks")

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

os.system("pip install -q lightgbm mlforecast")

print(f"✓ Setup complete — {os.getcwd()}")

# Module 5 — ML Forecasting with LightGBM
**Type:** [Code With Me]  
**Time:** 45 minutes  
**Job:** Prove that a feature-engineered ML pipeline beats the statistical baselines. Quantify the compute cost. Make an honest recommendation about whether the improvement justifies the infrastructure.

---

> **Instructor note:** This is the most technically dense module. Pace it in four blocks:
> - 5.1–5.3: Setup, panel load, feature framing (5 min)
> - 5.4–5.6: Configure MLForecast, define features, run CV (15 min, Code With Me)
> - 5.7–5.9: Conformal intervals, plot, score (12 min, Code With Me)
> - 5.10–5.12: Load full artifact, leaderboard update, enterprise cliff (8 min)
>
> The conformal interval construction in 5.8 is the hardest conceptual step. Budget 3 minutes for it.

---
## 5.1 — Setup
**[Watch Only]**

> **Instructor note (1 min):** Run silently. Note the new import: `MLForecast` and `LGBMRegressor`. Everything else is the same pattern as Modules 3 and 4.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams['figure.figsize'] = (14, 4)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from mlforecast.utils import PredictionIntervals

from config import (
    ARTIFACT_DIR,
    PARAMS_DIR,
    HORIZON,
    SEASON_LENGTH,
    N_WINDOWS,
    STEP_SIZE,
    REFIT,
    MICRO_SUBSET_N,
    WORKSHOP_SUBSET_N,
    RANDOM_SEED,
    USE_TUNED_PARAMS,
    USE_INTERVALS,
)
from src.checkpointing import load_checkpoint
from src.evaluation import score_forecasts
from src.schemas import validate_forecast
from src.workshop_utils import get_micro_subset

print("Setup complete.")
print(f"USE_TUNED_PARAMS = {USE_TUNED_PARAMS}")
print(f"USE_INTERVALS    = {USE_INTERVALS}")

Setup complete.
USE_TUNED_PARAMS = True
USE_INTERVALS    = True


---
## 5.2 — Load Panel and Micro Subset
**[Watch Only]**

> **Instructor note (30 sec):** Same pattern as Module 4. Load from the validated artifact, select top-N by volume.

In [2]:
panel = load_checkpoint("03_validated_panel")
micro, top_series = get_micro_subset(panel, n=MICRO_SUBSET_N)

print(f"Micro panel: {micro['unique_id'].nunique()} series, {len(micro):,} rows")


  ✓ RED PATH RECOVERY COMPLETE
    Artifact : 03_validated_panel
    File     : 03_validated_panel.parquet
    Rows     : 1,941,000

Micro panel: 50 series, 97,050 rows


---
## 5.3 — Why ML Forecasting?
**[Watch Only]**

> **Instructor note (2 min):** Make the cost argument explicit before writing any code. This section answers the question students always have: "if AutoETS is good, why bother with LightGBM?"

AutoETS is a strong univariate model. It sees one series at a time. It cannot learn patterns that span series — shared seasonal profiles, cross-SKU promotional lifts, or price elasticity signals.

**LightGBM via MLForecast is a global model.** It trains on all series simultaneously. Each series becomes a set of training rows. The model learns patterns across the entire category portfolio.

**The compute tradeoff is real:**  
AutoETS on 1,000 series takes seconds. LightGBM on 1,000 series with lag features takes longer and produces a model artifact you now have to version, deploy, and monitor. The accuracy gain must justify that operational cost.

**The three feature configurations we will build (sections 5.5A–5.5C):**

| Section | Feature type | What it adds |
|---|---|---|
| **5.5A — Base** | Lags (7, 14, 28), rolling mean, day-of-week + month | Demand history and calendar |
| **5.5B — Rich** | Extended lags (7–28), rolling stats, Fourier features, seasonal rolling | More complete numeric/date signal |
| **5.5C — Categorical** | M5 hierarchy labels (category, dept, state, store) as static features | Structural context no univariate model can see |

**5.5A** is the Code With Me baseline. **5.5B** and **5.5C** are Watch Only reference configurations you run in 5.9A and 5.9B for comparison.

Adding lags in a notebook is a one-liner. Maintaining a governed feature pipeline that recomputes these lags daily across 100,000 SKUs — with late data, backfills, and a product hierarchy lookup — is a data engineering project that takes months.

---
## 5.4 — Load LightGBM Parameters
**[Watch Only]**

> **Instructor note (1 min):** Load from `params/` based on the `USE_TUNED_PARAMS` toggle in config. The params files are pre-tested — do not change them live. Show students what is in the file.

In [3]:
params_file = "mlforecast_lgb_tuned.json" if USE_TUNED_PARAMS else "mlforecast_lgb_default.json"
params_path = PARAMS_DIR / params_file

with open(params_path) as f:
    lgb_params = json.load(f)

# Always lock the random seed from config — do not let params file override it
lgb_params["random_state"] = RANDOM_SEED

print(f"Loading params from: {params_file}")
for k, v in lgb_params.items():
    print(f"  {k:<22}: {v}")

Loading params from: mlforecast_lgb_tuned.json
  _comment              : AutoMLForecast Optuna-tuned params (default objective).
  n_estimators          : 675
  learning_rate         : 0.026585908882109703
  num_leaves            : 95
  min_child_samples     : 35
  subsample             : 0.8323930520817325
  colsample_bytree      : 0.9088540087875863
  reg_alpha             : 0.006250391643347274
  reg_lambda            : 0.011262001072312433
  random_state          : 42
  n_jobs                : -1
  verbosity             : -1


---
## 5.5A — Configure MLForecast (Base)
**[Code With Me — 3 lines]**

> **Instructor note (4 min):** This is the most important Code With Me cell in the module. Students fill in the `lags`, `lag_transforms`, and `date_features`. Walk through each argument:
> - `lags`: explicit integer list — `[7, 14, 28]`. These align to weekly, biweekly, and monthly demand patterns.
> - `lag_transforms`: rolling mean on lag 7, window 28 — smooths the weekly lag.
> - `date_features`: `dayofweek` and `month` — the two calendar signals that matter most for retail.
>
> Pause and check that everyone has the same config before running.
> Sections 5.5B and 5.5C define richer configs — we will compare all three in 5.9A and 5.9B.

In [4]:
# Configure MLForecast — all feature engineering happens here, not in a separate step

mlf = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq="D",
    lags=[7, 14, 28],                               # __FILL_IN__: weekly, biweekly, monthly lags
    lag_transforms={
        7: [RollingMean(window_size=28)],            # __FILL_IN__: rolling mean on lag 7, window 28
    },
    date_features=["dayofweek", "month"],            # __FILL_IN__: day-of-week and month
)

print("MLForecast configured.")
print(f"  Lags          : {mlf.ts.lags}")
print(f"  Date features : {mlf.ts.date_features}")
print(f"  Model         : {next(iter(mlf.models.values())).__class__.__name__}")

MLForecast configured.
  Lags          : [7, 14, 28]
  Date features : ['dayofweek', 'month']
  Model         : LGBMRegressor


---
### 5.5B — Feature Engineering Reference: Enhanced Numeric and Date Features
**[Watch Only]**

> **Instructor note (4 min):** This cell is a reference block — students do not fill anything in.
> Walk through each feature family and explain the intuition:
> - **More lags**: catch demand echoes at 1, 2, 3, 4 weeks + monthly
> - **Rolling statistics**: mean captures level, std captures volatility, min/max capture range, quantile captures distributional shape — all from the same lag window
> - **Fourier features**: encode seasonality as smooth sin/cos waves rather than integer labels. `dayofweek=3` gives the model no information about adjacency to days 2 and 4; `sin(2π·dow/7)` does. Two terms per period (k=1) is usually enough for weekly.
> - **Seasonal rolling mean**: average of the same weekday over the past N weeks — directly encodes the weekly pattern without relying on the model to discover it
>
> The base config in 5.5A is deliberately minimal. This block shows the full numeric and date toolkit.
> 5.5C adds one more layer — categorical static features — on top of this.

**Why Fourier over integer day-of-week?**
Integer labels (`dayofweek=3`) are ordinal — the model can learn a non-linear mapping, but only if it sees enough data.
Fourier features encode the circular structure of the week *in the input*, so the model needs fewer samples to learn the pattern.
On sparse series or short histories, Fourier features often outperform integer encoding.

A plain `dayofweek` column treats the days as numbers:

- Monday = 0
- Tuesday = 1
- ...
- Sunday = 6

That is convenient, but it hides the real structure. The week is **cyclical**, not linear. Sunday and Monday are next to each other in real life, but as integers they look far apart: `6` and `0`.

A model can eventually learn that pattern, but it has to discover it from the data.

Fourier features give the model that circular weekly structure upfront. They represent the week as smooth repeating sine/cosine waves, so the model can more easily learn patterns like:

- weekends behave differently from weekdays,
- demand rises into Friday,
- Monday resets after Sunday,
- weekly seasonality repeats.

So the practical reason is:

> Fourier features make weekly seasonality easier for the model to learn, especially when the history is short, sparse, or noisy.

In [5]:
# 5.5b — Feature Engineering Reference: Richer MLForecast Feature Sets
# This cell is a REFERENCE — swap the mlf definition in 5.5 to experiment.
# Each feature family is shown independently, then combined at the bottom.

import numpy as np
from mlforecast import MLForecast
from mlforecast.lag_transforms import (
    RollingMean,
    RollingStd,
    RollingMin,
    RollingMax,
    RollingQuantile,
    SeasonalRollingMean,
)

# ── 1. Lag list ───────────────────────────────────────────────────────────────
# Explicit lags: 1 week, 2 weeks, 3 weeks, 4 weeks, ~1 month
LAGS = [7, 14, 21, 28]

# ── 2. Rolling statistics on lag 7 ───────────────────────────────────────────
# window_size=4 weeks → statistics over a full month of same-weekday observations
# RollingQuantile(p=0.5) is the rolling median — robust to outlier demand spikes
ROLLING_28 = [
    RollingMean(window_size=28),
    RollingStd(window_size=28),
    RollingMin(window_size=28),
    RollingMax(window_size=28),
    RollingQuantile(p=0.25, window_size=28),  # lower quartile
    RollingQuantile(p=0.75, window_size=28),  # upper quartile
]

# ── 3. Seasonal rolling mean ──────────────────────────────────────────────────
# SeasonalRollingMean(season_length=7, window_size=4):
#   average of the same weekday over the past 4 weeks.
#   Directly encodes the weekly demand profile without the model needing to
#   discover it from raw lags.
SEASONAL_ROLLING = [
    SeasonalRollingMean(season_length=7, window_size=4),   # 4-week same-weekday mean
    SeasonalRollingMean(season_length=7, window_size=8),   # 8-week same-weekday mean
]

# ── 4. Fourier date features (via MLForecast date_features callables) ─────────
# MLForecast's date_features accepts any named callable that takes a
# pandas DatetimeIndex and returns a numeric array.
# Two Fourier terms per season (k=1) encode the weekly cycle as a smooth wave.
# k=2 adds a second harmonic — captures asymmetric within-week patterns.
#
# Why this instead of dayofweek integer?
# Integer encoding: model must learn non-linearity from data.
# Fourier encoding: circular structure is baked into the feature — fewer samples needed.

def fourier_sin1_weekly(dates):
    """First Fourier sine term for weekly seasonality (period=7)."""
    return np.sin(2 * np.pi * 1 * dates.dayofweek / 7)

def fourier_cos1_weekly(dates):
    """First Fourier cosine term for weekly seasonality (period=7)."""
    return np.cos(2 * np.pi * 1 * dates.dayofweek / 7)

def fourier_sin2_weekly(dates):
    """Second Fourier sine term — captures asymmetric weekly patterns."""
    return np.sin(2 * np.pi * 2 * dates.dayofweek / 7)

def fourier_cos2_weekly(dates):
    """Second Fourier cosine term — captures asymmetric weekly patterns."""
    return np.cos(2 * np.pi * 2 * dates.dayofweek / 7)

def fourier_sin1_annual(dates):
    """First Fourier sine term for annual seasonality (period=365.25)."""
    return np.sin(2 * np.pi * 1 * dates.dayofyear / 365.25)

def fourier_cos1_annual(dates):
    """First Fourier cosine term for annual seasonality (period=365.25)."""
    return np.cos(2 * np.pi * 1 * dates.dayofyear / 365.25)

DATE_FEATURES = [
    "month",
    fourier_sin1_weekly,
    fourier_cos1_weekly,
    fourier_sin2_weekly,
    fourier_cos2_weekly,
    fourier_sin1_annual,
    fourier_cos1_annual,
]

# ── 5. Full config ────────────────────────────────────────────────────────────
mlf_rich = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq="D",
    lags=LAGS,
    lag_transforms={
        7:  ROLLING_28 + SEASONAL_ROLLING,
        14: [RollingMean(window_size=28)],
    },
    date_features=DATE_FEATURES,
)

# Inspect the feature set without fitting
feature_names = mlf_rich.ts.transforms
print("Rich MLForecast feature set:")
print(f"  Lags                 : {LAGS}")
print(f"  Lag-7 transforms     : {len(ROLLING_28 + SEASONAL_ROLLING)} features")
print(f"    Rolling stats      : RollingMean, RollingStd, RollingMin, RollingMax, Q25, Q75")
print(f"    Seasonal rolling   : SeasonalRollingMean x2 (4-week, 8-week same-weekday)")
print(f"  Lag-14 transforms    : RollingMean(28)")
print(f"  Date features        : {len(DATE_FEATURES)} features")
print(f"    Calendar           : month")
print(f"    Fourier weekly k=2 : sin1, cos1, sin2, cos2")
print(f"    Fourier annual k=1 : sin1, cos1")
print()
print("To use this config for CV, replace `mlf` with `mlf_rich` in cell 5.6.")
print("Expected: +1-3 wMAPE improvement on datasets with strong weekly + annual seasonality.")

Rich MLForecast feature set:
  Lags                 : [7, 14, 21, 28]
  Lag-7 transforms     : 8 features
    Rolling stats      : RollingMean, RollingStd, RollingMin, RollingMax, Q25, Q75
    Seasonal rolling   : SeasonalRollingMean x2 (4-week, 8-week same-weekday)
  Lag-14 transforms    : RollingMean(28)
  Date features        : 7 features
    Calendar           : month
    Fourier weekly k=2 : sin1, cos1, sin2, cos2
    Fourier annual k=1 : sin1, cos1

To use this config for CV, replace `mlf` with `mlf_rich` in cell 5.6.
Expected: +1-3 wMAPE improvement on datasets with strong weekly + annual seasonality.


---
### 5.5C — Static Categorical Features: M5 Product Hierarchy
**[Watch Only]**

> **Instructor note (3 min):** This is the key differentiator of global ML models over univariate models.
>
> Every model we have run so far — Naive, SeasonalNaive, AutoETS, Chronos — processes each series in isolation. It cannot learn: *"this is a FOODS item in Wisconsin — what does that mean for demand patterns compared to a HOBBIES item in California?"*
>
> LightGBM is the first model in our stack that can use structural information that spans series. The M5 unique_id encodes a full product-store hierarchy. We extract it and pass it as **static features** — the same value for every row of a given series.
>
> Key points to make:
> - Static features are NOT time-varying — they are fixed properties of the series. MLForecast passes them to LightGBM as constant columns at every training row.
> - `static_features` is passed to `cross_validation()` as a keyword argument. MLForecast forwards it to the internal `fit()` call.
> - LightGBM requires numeric inputs. We integer-encode each categorical with pandas `.cat.codes` — stable within this dataset.
> - In production, static features come from a product master table or hierarchy lookup — not from parsing series IDs.
>
> Ask the room: "If your dataset doesn't encode the hierarchy in the series ID, how do you add this?" (Answer: join it from a product master before fit. The feature pipeline owns this join.)

In [ ]:
# 5.5C — Static Categorical Features: extracted from M5 unique_id hierarchy
# ─────────────────────────────────────────────────────────────────────────────
# M5 unique_id format: {CATEGORY}_{DEPT_NUM}_{ITEM_NUM}_{STATE_ID}_{STORE_NUM}
# Example: FOODS_2_360_WI_2
#   → category = FOODS     (3 categories: FOODS, HOBBIES, HOUSEHOLD)
#   → dept     = FOODS_2   (7 departments across all categories)
#   → state    = WI        (3 states: CA, TX, WI)
#   → store    = WI_2      (10 stores total across all states)
#
# These are STATIC per series — the same value for every timestamp of a given unique_id.
# MLForecast attaches them as constant columns to every LightGBM training row.

def add_m5_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract and integer-encode M5 hierarchy labels from unique_id.
    LightGBM requires numeric inputs — pandas Categorical codes provide
    stable integer encoding within this dataset.
    """
    parts = df["unique_id"].str.split("_")
    df = df.copy()
    df["cat_id"]   = parts.str[0]                    # FOODS, HOBBIES, HOUSEHOLD
    df["dept_id"]  = parts.str[:2].str.join("_")     # FOODS_1, FOODS_2, HOBBIES_1 ...
    df["state_id"] = parts.str[3]                    # CA, TX, WI
    df["store_id"] = parts.str[3:5].str.join("_")   # CA_1, CA_2, TX_1 ...
    for col in ["cat_id", "dept_id", "state_id", "store_id"]:
        df[col] = df[col].astype("category").cat.codes  # integer codes: 0, 1, 2 ...
    return df

STATIC_FEATURES = ["cat_id", "dept_id", "state_id", "store_id"]

# Create category-augmented micro panel
micro_cat = add_m5_categoricals(micro)

print("Categorical feature extraction:")
print(f"  Panel rows preserved : {len(micro_cat):,}")
for col in STATIC_FEATURES:
    vals = sorted(micro_cat[col].unique().tolist())
    print(f"  {col:<12} : {vals}  ({len(vals)} distinct encoded values)")
print()

# mlf_cat: same rich feature set as 5.5B, PLUS static categoricals
# The static_features argument is passed to cross_validation() — not to the constructor.
# MLForecast forwards it to fit() internally for each CV window.
mlf_cat = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq="D",
    lags=LAGS,
    lag_transforms={
        7:  ROLLING_28 + SEASONAL_ROLLING,
        14: [RollingMean(window_size=28)],
    },
    date_features=DATE_FEATURES,
)

print("MLForecast-Cat configured.")
print(f"  Numeric/date features : same as 5.5B (lags, rolling stats, Fourier)")
print(f"  Static cat features   : {STATIC_FEATURES}")
print(f"  Usage: mlf_cat.cross_validation(df=micro_cat, ..., static_features=STATIC_FEATURES)")

**Expected output:**
```
MLForecast configured.
  Lags          : [7, 14, 28]
  Date features : ['dayofweek', 'month']
  Model         : LGBMRegressor
```

---
## 5.6 — Run Cross-Validation on the Micro Subset
**[Watch Only]**

> **Instructor note (3 min):** Run and let it execute. LightGBM on 50 series with these features should complete in 15–25 seconds. While it runs, explain the recursive forecasting strategy: MLForecast feeds its own predictions back as lag inputs for each step in the horizon. This is why lag features create dependency chains.

In [6]:
%%time
# Cross-validation on micro subset
# Target runtime: < 30 seconds on Colab CPU
# PredictionIntervals requires refit=True — controlled via USE_INTERVALS in config.py

cv_ml_micro = mlf_rich.cross_validation(
    df=micro,
    h=HORIZON,
    n_windows=N_WINDOWS,
    step_size=STEP_SIZE,
    refit=True if USE_INTERVALS else REFIT,                                                              # __FILL_IN__: True when using PredictionIntervals, REFIT otherwise
    prediction_intervals=PredictionIntervals(n_windows=N_WINDOWS, h=HORIZON) if USE_INTERVALS else None, # __FILL_IN__: Nixtla-native conformal intervals
    level=[80] if USE_INTERVALS else None,                                                               # __FILL_IN__: 80% prediction interval
)

print(f"ML CV complete. Shape: {cv_ml_micro.shape}")
print(f"Columns: {list(cv_ml_micro.columns)}")
print(cv_ml_micro.head(3).to_string())

ML CV complete. Shape: (4200, 7)
Columns: ['unique_id', 'ds', 'cutoff', 'y', 'LGBMRegressor', 'LGBMRegressor-lo-80', 'LGBMRegressor-hi-80']
          unique_id         ds     cutoff    y  LGBMRegressor  LGBMRegressor-lo-80  LGBMRegressor-hi-80
0  FOODS_2_360_WI_2 2016-02-29 2016-02-28  0.0       5.623603             1.824491             9.422715
1  FOODS_2_360_WI_2 2016-03-01 2016-02-28  0.0       5.757014             1.622279             9.891748
2  FOODS_2_360_WI_2 2016-03-02 2016-02-28  0.0       5.520811             1.365172             9.676450
CPU times: total: 7min 59s
Wall time: 17.2 s


**Expected output:**
```
ML CV complete. Shape: (4200, 7)
Columns: ['ds', 'cutoff', 'y', 'LGBMRegressor', 'LGBMRegressor-lo-80', 'LGBMRegressor-hi-80']
```

> **Instructor note:** MLForecast now returns interval columns directly — `LGBMRegressor-lo-80` and `LGBMRegressor-hi-80`. These are calibrated using held-out cross-validation windows via `PredictionIntervals`, not in-sample residuals. The reshape in 5.7 just renames these columns to the schema format.

---
## 5.7 — Reshape MLForecast CV Output
**[Code With Me — 2 lines]**

> **Instructor note (3 min):** MLForecast returns wide format with one column per model plus interval columns named `{model}-lo-80` and `{model}-hi-80`. The reshape function standardizes this to the forecast schema. Two fill-ins: rename the lo/hi columns to `lo_80` / `hi_80`.
>
> Key point to make: the intervals come from `PredictionIntervals` — MLForecast holds out additional CV windows internally to calibrate the conformal bounds. This is more statistically sound than computing percentiles of in-sample residuals, because calibration and evaluation use separate data.

In [ ]:
# Reshape MLForecast wide output to forecast schema
def reshape_mlforecast_cv(
    cv_df: pd.DataFrame,
    model_col: str,
    stage: str,
    model_name: str = "LightGBM",
) -> pd.DataFrame:
    """
    Reshape MLForecast wide CV output to forecast schema.
    Interval columns (lo_80, hi_80) come from PredictionIntervals — Nixtla-native conformal calibration.

    Parameters
    ----------
    cv_df      : Wide CV output from MLForecast.cross_validation()
    model_col  : Name of the point forecast column (e.g. "LGBMRegressor")
    stage      : Stage label for the forecast schema ("ml")
    model_name : Display name in the model column (default: "LightGBM")
    """
    df = cv_df.reset_index().copy()
    df = df.rename(columns={
        model_col:              "y_hat",
        f"{model_col}-lo-80":  "lo_80",   # __FILL_IN__: rename native lo column to schema name
        f"{model_col}-hi-80":  "hi_80",   # __FILL_IN__: rename native hi column to schema name
    })
    df["model"] = model_name
    df["stage"] = stage
    df["y_hat"] = df["y_hat"].clip(lower=0)
    df["lo_80"] = df["lo_80"].clip(lower=0)
    df["hi_80"] = df["hi_80"].clip(lower=0)

    return df[["unique_id", "ds", "y", "model", "y_hat", "lo_80", "hi_80", "cutoff", "stage"]]


# cv_ml_micro came from mlf_rich.cross_validation() in 5.6 → model_name="LightGBM-Rich"
ml_micro = reshape_mlforecast_cv(cv_ml_micro, model_col="LGBMRegressor", stage="ml", model_name="LightGBM-Rich")

print(f"Reshaped ML forecasts: {ml_micro.shape}")
print(f"Columns: {list(ml_micro.columns)}")
print(f"Interval sample (lo_80, y_hat, hi_80):")
print(ml_micro[["lo_80", "y_hat", "hi_80"]].describe().round(2).to_string())

**Expected output:**
```
Reshaped ML forecasts: (4200, 9)
Columns: ['unique_id', 'ds', 'y', 'model', 'y_hat', 'lo_80', 'hi_80', 'cutoff', 'stage']
```

---
## 5.8 — Plot: LightGBM vs Best Baseline
**[Watch Only]**

> **Instructor note (2 min):** Load the SeasonalNaive micro forecast from the baseline micro artifact for visual comparison. Show both on screen. Ask the room: does LightGBM visually improve over SeasonalNaive? The answer is often "not obviously" — which is exactly why we score it.

In [ ]:
# Compare LightGBM-Rich vs SeasonalNaive on a single series
sample_uid = top_series[0]
sample_cut = ml_micro["cutoff"].unique()[-1]

actuals   = panel[panel["unique_id"] == sample_uid].set_index("ds")["y"]
plot_start = sample_cut - pd.Timedelta(days=42)

lgbm_fcast = ml_micro[
    (ml_micro["unique_id"] == sample_uid) &
    (ml_micro["cutoff"]    == sample_cut)
].set_index("ds")

# Load micro baselines for SeasonalNaive comparison
try:
    baseline_micro_path = ARTIFACT_DIR / "04_baseline_micro_forecasts.parquet"
    baseline_micro = pd.read_parquet(baseline_micro_path)
    snaive_fcast = baseline_micro[
        (baseline_micro["unique_id"] == sample_uid) &
        (baseline_micro["cutoff"]    == sample_cut) &
        (baseline_micro["model"]     == "SeasonalNaive")
    ].set_index("ds")
    has_baseline = True
except Exception:
    has_baseline = False

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(actuals[actuals.index >= plot_start].index,
        actuals[actuals.index >= plot_start].values,
        color="#333", linewidth=1.0, label="Actual", zorder=3)

ax.plot(lgbm_fcast.index, lgbm_fcast["y_hat"],
        color="#7B1FA2", linewidth=1.5, linestyle="--", label="LightGBM-Rich", zorder=4)
ax.fill_between(lgbm_fcast.index, lgbm_fcast["lo_80"], lgbm_fcast["hi_80"],
                alpha=0.15, color="#7B1FA2", label="LightGBM-Rich 80% interval")

if has_baseline and len(snaive_fcast) > 0:
    ax.plot(snaive_fcast.index, snaive_fcast["y_hat"],
            color="#1E88E5", linewidth=1.2, linestyle=":", label="SeasonalNaive", zorder=3)

ax.axvline(sample_cut, color="#999", linestyle=":", linewidth=1)
ax.set_title(f"LightGBM-Rich vs SeasonalNaive — {sample_uid} (Window 3)", fontsize=11)
ax.set_ylabel("Units sold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Expected output:** LightGBM forecast with shaded 80% interval alongside SeasonalNaive. The interval width communicates model confidence — compare it to the SeasonalNaive line visually.

---
## 5.9A — Micro Subset Comparison: Base vs Rich vs Categorical
**[Watch Only]**

> **Instructor note (4 min):** This cell runs all three configs on the 50-series micro subset and produces a comparison table.
>
> The point of this section is NOT to pick a winner. It is to demonstrate that 50 series is not enough to reliably distinguish three feature sets.
>
> Walk through the table when it prints. Then say this explicitly:
> *"The ordering you see here is likely noise. 50 series is a highly selected, low-variance sample of top sellers. If any config looks worse on micro, that doesn't mean it performs worse at scale — it means the selection bias is strong enough to invert the real signal."*
>
> Immediately contrast with 5.9B. The full-panel result is the one to use for decisions.
>
> mlf_rich already ran in 5.6 — its result is reused here. This cell additionally runs mlf (base) and mlf_cat on micro.

In [ ]:
%%time
# 5.9A — Micro Subset Comparison: Base vs Rich vs Categorical
# ─────────────────────────────────────────────────────────────
# mlf_rich already ran in 5.6. Reuse cv_ml_micro (reshape already done in 5.7 → ml_micro).
# This cell additionally runs mlf (base) and mlf_cat on micro, then compares all three.

def _get_score(scores_df, metric):
    row = scores_df[scores_df["metric"] == metric]
    return row.iloc[0]["score"] if len(row) > 0 else float("nan")

# ── mlf_rich result from 5.6/5.7 — validate and score ────────────────────────
ml_rich_micro_val = validate_forecast(ml_micro, artifact_name="05_ml_rich_micro")  # __FILL_IN__
scores_rich = score_forecasts(ml_rich_micro_val, subset_name=f"micro_{MICRO_SUBSET_N}")

# ── Base config (mlf) on micro — quick run ────────────────────────────────────
cv_base_micro = mlf.cross_validation(
    df=micro,
    h=HORIZON,
    n_windows=N_WINDOWS,
    step_size=STEP_SIZE,
    refit=True if USE_INTERVALS else REFIT,
    prediction_intervals=PredictionIntervals(n_windows=N_WINDOWS, h=HORIZON) if USE_INTERVALS else None,
    level=[80] if USE_INTERVALS else None,
)
ml_base_micro = reshape_mlforecast_cv(cv_base_micro, model_col="LGBMRegressor", stage="ml", model_name="LightGBM")
ml_base_micro_val = validate_forecast(ml_base_micro, artifact_name="05_ml_base_micro")
scores_base = score_forecasts(ml_base_micro_val, subset_name=f"micro_{MICRO_SUBSET_N}")

# ── Categorical config (mlf_cat) on micro ─────────────────────────────────────
cv_cat_micro = mlf_cat.cross_validation(
    df=micro_cat,
    h=HORIZON,
    n_windows=N_WINDOWS,
    step_size=STEP_SIZE,
    refit=True if USE_INTERVALS else REFIT,
    prediction_intervals=PredictionIntervals(n_windows=N_WINDOWS, h=HORIZON) if USE_INTERVALS else None,
    level=[80] if USE_INTERVALS else None,
    static_features=STATIC_FEATURES,
)
ml_cat_micro = reshape_mlforecast_cv(cv_cat_micro, model_col="LGBMRegressor", stage="ml", model_name="LightGBM-Cat")
ml_cat_micro_val = validate_forecast(ml_cat_micro, artifact_name="05_ml_cat_micro")
scores_cat = score_forecasts(ml_cat_micro_val, subset_name=f"micro_{MICRO_SUBSET_N}")

# ── Comparison table ───────────────────────────────────────────────────────────
W = 24
print(f"Micro Subset Comparison ({MICRO_SUBSET_N} series, {N_WINDOWS} windows):")
print(f"  {'Config':<{W}} {'wMAPE':>8}  {'Bias':>8}  {'IntervalScore_80':>16}")
print(f"  {'-'*66}")
for label, scores in [
    ("Base — 5.5A",        scores_base),
    ("Rich — 5.5B",        scores_rich),
    ("Categorical — 5.5C", scores_cat),
]:
    wmape  = _get_score(scores, "wMAPE")
    bias   = _get_score(scores, "Bias")
    iscore = _get_score(scores, "IntervalScore_80")
    print(f"  {label:<{W}} {wmape:>8.4f}  {bias:>+8.4f}  {iscore:>16.4f}")
print()
print(f"Key point: {MICRO_SUBSET_N} series is not enough to reliably distinguish these configs.")
print("Any ordering here is likely noise. See 5.9B for the full-panel view.")

---
## 5.9B — Full Panel Comparison: Base vs Rich vs Categorical
**[Watch Only]**

> **Instructor note (3 min):** This is the reliable view. Three configs across 1,000 series and 3 CV windows.
>
> Contrast the ordering here with 5.9A. The micro result likely showed near-identical scores or a different ordering. The full-panel result is the one that actually reflects the signal — 1,000 series gives the model enough variety to learn department and state effects that are invisible in 50 top-sellers.
>
> For categorical features specifically: M5 has strong dept and state effects. Items in FOODS_1 in California behave very differently from HOBBIES_2 in Wisconsin. A global model that can see this structural information has a systematic advantage over one that cannot.
>
> Framing for the room: "If your dataset doesn't have a clean categorical hierarchy in the series IDs, this is still achievable — you join it from a product master table before fit. The feature pipeline owns that join. The model sees it as just another column."
>
> Secondary lesson: feature engineering (+X%) typically beats hyperparameter tuning (~0.5%). Categorical features add another layer on top. The ordering of investment should be: features before tuning, structure before complexity.

In [ ]:
# 5.9B — Full Panel Comparison: Base vs Rich vs Categorical (Precomputed)
# ─────────────────────────────────────────────────────────────────────────
# Load all three precomputed full-panel artifacts (WORKSHOP_SUBSET_N series).
# These results are stable across 1,000 series — use them for production decisions.

try:
    ml_full_base = load_checkpoint("05_ml_forecasts")
    ml_full_rich = load_checkpoint("05_ml_rich_forecasts")
    ml_full_cat  = load_checkpoint("05_ml_cat_forecasts")

    subset_label     = f"workshop_{WORKSHOP_SUBSET_N}"
    scores_full_base = score_forecasts(ml_full_base, subset_name=subset_label)
    scores_full_rich = score_forecasts(ml_full_rich, subset_name=subset_label)
    scores_full_cat  = score_forecasts(ml_full_cat,  subset_name=subset_label)

    W = 24
    print(f"Full Panel Comparison ({WORKSHOP_SUBSET_N:,} series, {N_WINDOWS} windows — RELIABLE):")
    print(f"  {'Config':<{W}} {'wMAPE':>8}  {'Bias':>8}  {'IntervalScore_80':>16}")
    print(f"  {'-'*66}")
    for label, scores in [
        ("Base — 5.5A",        scores_full_base),
        ("Rich — 5.5B",        scores_full_rich),
        ("Categorical — 5.5C", scores_full_cat),
    ]:
        wmape  = _get_score(scores, "wMAPE")
        bias   = _get_score(scores, "Bias")
        iscore = _get_score(scores, "IntervalScore_80")
        print(f"  {label:<{W}} {wmape:>8.4f}  {bias:>+8.4f}  {iscore:>16.4f}")

    print()
    print("Key lesson: micro result is noise. Full panel is the signal.")
    print("Categorical features add structural context no univariate model can access.")

except FileNotFoundError as e:
    print(f"Artifact not found: {e}")
    print("Run: python src/build_offline_artifacts.py --stages ml ml_rich ml_cat")



> **Instructor note:** Do not read the micro score as final. The micro subset is the top-50 by volume — the easiest 50 to forecast. The full-subset score will be higher (worse). This is a known selection bias.

---
## 5.10 — Full Leaderboard: All ML Configs vs Baselines
**[Watch Only]**

> **Instructor note (2 min):** Build the leaderboard with all three LightGBM configs alongside the baselines. This shows the full progression: baseline → numeric features → categorical features.
>
> The question for the room: "Categorical adds X wMAPE points. Your product team maintains a department and store hierarchy already — it lives in your ERP. The feature pipeline cost is one join. Is that worth it?" (Almost always yes, if the hierarchy is clean and stable.)
>
> Use the `ml_full_base/rich/cat` variables already loaded in 5.9B. If 5.9B's try/except caught an error, the variables won't be in scope — the cell below handles that gracefully.

In [ ]:
# 5.10 — Build the full leaderboard: all three ML configs + statistical baselines
# ─────────────────────────────────────────────────────────────────────────────
# Reuse artifacts already loaded in 5.9B. Load baselines fresh.

subset_label = f"workshop_{WORKSHOP_SUBSET_N}"

# Load baselines
baseline_full   = load_checkpoint("04_baseline_forecasts")
baseline_scores = score_forecasts(baseline_full, subset_name=subset_label)

# ML scores — reuse from 5.9B if in scope, otherwise reload
try:
    _base_ok = "scores_full_base" in dir() and scores_full_base is not None
except NameError:
    _base_ok = False

if not _base_ok:
    ml_full_base = load_checkpoint("05_ml_forecasts")
    ml_full_rich = load_checkpoint("05_ml_rich_forecasts")
    ml_full_cat  = load_checkpoint("05_ml_cat_forecasts")
    scores_full_base = score_forecasts(ml_full_base, subset_name=subset_label)
    scores_full_rich = score_forecasts(ml_full_rich, subset_name=subset_label)
    scores_full_cat  = score_forecasts(ml_full_cat,  subset_name=subset_label)

from src.evaluation import build_leaderboard
leaderboard_5 = build_leaderboard([baseline_scores, scores_full_base, scores_full_rich, scores_full_cat])

print("Updated leaderboard — all models (sorted by wMAPE):")
display_cols = ["model", "wMAPE", "Bias"]
wmape_lb = (
    leaderboard_5[display_cols]
    .dropna(subset=["wMAPE"])
    .sort_values("wMAPE")
    if "wMAPE" in leaderboard_5.columns
    else leaderboard_5
)
print(wmape_lb.to_string(index=False))

**Expected output:**
```
Updated leaderboard — all models (sorted by wMAPE):
      model    wMAPE      Bias
LightGBM-Cat   0.XXXX   +0.XXXX
LightGBM-Rich  0.XXXX   +0.XXXX
    LightGBM   0.XXXX   +0.XXXX
     AutoETS   0.XXXX   +0.XXXX
 Chronos-t5-mini 0.XXXX  -0.XXXX
SeasonalNaive   0.XXXX   -0.XXXX
       Naive   0.XXXX   +0.XXXX
```

> **Instructor note:** The three LightGBM variants should show a clear progression: Base < Rich < Categorical in terms of wMAPE. If the ordering differs, ask the room why — the answer is usually that the micro result they saw earlier was misleading, or that M5 department effects are weaker than expected on this subset.

---
## 5.11 — Save Micro ML Artifacts
**[Watch Only]**

> **Instructor note (30 sec):** Save all three micro artifacts for optional take-home analysis. The full-subset artifacts were already loaded from precomputed checkpoints.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

micro_artifacts = {
    "05_ml_base_micro_forecasts.parquet": ml_base_micro_val,
    "05_ml_rich_micro_forecasts.parquet": ml_rich_micro_val,
    "05_ml_cat_micro_forecasts.parquet":  ml_cat_micro_val,
}
for fname, df_val in micro_artifacts.items():
    path = ARTIFACT_DIR / fname
    df_val.to_parquet(path, index=False)
    print(f"  ✓ Micro artifact saved : {fname} ({len(df_val):,} rows)")

print()
print("Full-panel artifacts (precomputed checkpoints):")
print(f"  ✓ 05_ml_forecasts.parquet          ({len(ml_full_base):,} rows) — LightGBM base")
print(f"  ✓ 05_ml_rich_forecasts.parquet      ({len(ml_full_rich):,} rows) — LightGBM-Rich")
print(f"  ✓ 05_ml_cat_forecasts.parquet       ({len(ml_full_cat):,} rows)  — LightGBM-Cat")

---
## 5.12 — The Enterprise Cliff
**[Watch Only]**

> **Instructor note (2 min):** This is the authority hook. Say the first paragraph slowly — it is the line that differentiates practitioners from notebook runners.

Adding lags in a notebook is a one-liner. **Orchestrating a scalable feature pipeline across 100,000 SKUs — with categorical lookups, backfills, and audit trails — is where enterprise forecasting systems diverge from standalone scripts.**

What we simplified:

**Lag computation at scale:**  
MLForecast computes lags on the fly during CV. In production, lags are computed once, stored in a versioned feature store, and served to the model at inference time. Late-arriving data invalidates precomputed lags — your pipeline needs a backfill strategy.

**Categorical features in production:**  
We extracted categoricals from the series ID. In production, static features come from a product master table or store hierarchy in your ERP. The feature pipeline owns the join — and must handle new products, discontinued SKUs, and store openings that weren't in the training data. A model that has never seen a new store's state encoding will return a silent garbage prediction.

**Conformal intervals at scale:**  
`PredictionIntervals` uses held-out cross-validation windows to calibrate the conformal bounds — calibration data is separate from training data. That said, the calibration set is still drawn from the same historical distribution. On series with sudden trend shifts or promotional events, the historical error distribution will not match future errors, and the intervals will undercover.

**The ROI question:**  
LightGBM-Cat beat AutoETS by some margin on wMAPE. The question your CFO asks is: how many basis points of inventory reduction does that margin translate to? If the answer is less than the annual engineering cost of maintaining a feature pipeline with a product hierarchy lookup, AutoETS is the right production choice.

---

> **Instructor note — transition:** "We have a working ML pipeline with three feature configurations on the leaderboard. Module 8 will load `05_ml_cat_forecasts` as the representative LightGBM result for the final evaluation. One more model to add — a global neural model."
>
> Open `instructor_notebooks/06_deep_learning.ipynb`.